# 11. Integrarea LLM - explicatii in limbaj natural

**Etapa IV: Integrarea modelelor de limbaj (LLM)**

**Scop:** ultima componenta AI a platformei. Pana acum avem **predictii** (Etapa II) si **recomandari de optimizare** (Etapa III), dar acestea sunt cifre - metrici, vectori de dispatch, unghiuri. Un operator uman are nevoie de **explicatii in limbaj natural**: "ce inseamna" si "ce sa fac". Aici inchidem bucla decizionala, traducand rezultatele numerice in text inteligibil.

Folosim un model de limbaj de tip **text-to-text** din ecosistemul HuggingFace (flan-t5), open-source si rulabil local, plus un generator determinist pe sabloane pentru robustete si reproductibilitate.

## Setup

In [1]:
import sys
from pathlib import Path
import warnings
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.llm_integration import insights as ins
from src.optimization.optimizer import BatteryConfig, solve_battery_dispatch
warnings.filterwarnings("ignore")
print("Setup OK.")

Setup OK.


## 1. Concepte: cum functioneaza un model de limbaj

### Tokeni
Un model de limbaj nu vede cuvinte, ci **tokeni** - bucati de text (cuvinte sau fragmente de cuvinte) transformate in numere. Propozitia "pretul creste" devine, de exemplu, secventa [pret, ul, creste] -> [1820, 7, 9032]. Modelul lucreaza cu aceste numere.

### Embeddings
Fiecare token este transformat intr-un **vector** de cateva sute de numere (un embedding) care ii codifica sensul. Cuvinte cu sens apropiat au vectori apropiati. Astfel modelul "intelege" relatii: vectorul pentru "scump" e aproape de "costisitor" si departe de "ieftin".

### Transformer
Arhitectura **transformer** (introdusa in 2017, "Attention is all you need") proceseaza toti tokenii simultan si invata, prin mecanismul de **atentie**, care cuvinte sunt relevante unele pentru altele in context. Aceasta capacitate de a cantari contextul face transformerele foarte puternice pentru limbaj.

### Model instruction-tuned: flan-t5
**flan-t5** (Google) este un transformer de tip **text-to-text**: primeste text si produce text. A fost antrenat suplimentar pe instructiuni ("instruction-tuned"), deci raspunde bine la cereri formulate ca sarcini: "Explain...", "Summarize...", "Translate...". Varianta `base` e mica (ruleaza pe CPU), potrivita pentru o platforma fara infrastructura scumpa.

### Prompt engineering
Calitatea raspunsului depinde de **cum formulam cererea** (prompt-ul). Un prompt bun ofera context, datele relevante si sarcina clara. Comparam: "spune ceva despre pret" (vag) vs "Esti analist energetic. Explica pe scurt acest rezultat: R2=0.97, MAPE=2.5%" (specific).

### Parametri de generare
- **max_new_tokens**: cate cuvinte (tokeni) maxim sa genereze - controleaza lungimea.
- **temperature**: cat de "creativ" sa fie. 0 = determinist (alege mereu cel mai probabil cuvant); valori mari = mai variat, dar si mai riscant.

## 2. Arhitectura componentei: LLM + fallback determinist

Modulul `src/llm_integration/insights.py` ofera doua nivele complementare:
1. **flan-t5** (HuggingFace) - genereaza/rafineaza text. La prima rulare descarca modelul (~1 GB), deci ruleaza pe masina cu resurse (PyCharm local).
2. **Generator determinist pe sabloane** - construieste explicatii in romana direct din cifre, fara model. E mereu disponibil, reproductibil si serveste atat ca **fallback** (cand modelul nu e instalat), cat si ca **baseline** de comparatie.

Aceasta arhitectura cu degradare eleganta e o practica de productie: sistemul produce mereu un raspuns util, chiar daca modelul greu nu e disponibil.

## 3. Explicarea unei predictii (rezultate reale din Etapa II)

In [2]:
# Incarcam metricile reale ale modelului castigator pentru Spania
csv = PROJECT_ROOT / "reports" / "ml_comparison_spania_databricks.csv"
metrics_spania = {"rmse": 2.00, "mae": 1.49, "r2": 0.9696, "mape": 2.48}
if csv.exists():
    dfm = pd.read_csv(csv).sort_values("r2", ascending=False).iloc[0]
    metrics_spania = {k: float(dfm[k]) for k in ["rmse", "mae", "r2", "mape"]}

res = ins.explain_prediction(
    dataset="pretul energiei in Spania",
    target="pretul orar (EUR/MWh)",
    metrics=metrics_spania, unit="EUR/MWh",
    top_features=["price actual_lag_1", "price day ahead", "price actual_roll_mean_3"],
)
print("=== Explicatie generata (determinist) ===\n")
print(res["template"])

=== Explicatie generata (determinist) ===

Modelul pentru pretul energiei in Spania prezice pretul orar (EUR/MWh) cu o precizie foarte buna, utila in practica. Coeficientul de determinare R2 este 0.970, adica modelul explica 97.0% din variatia tintei. Eroarea medie absoluta este de aproximativ 1.49 EUR/MWh, iar eroarea procentuala medie (MAPE) de 2.5%. Cei mai influenti factori in predictie sunt: price actual_lag_1, price day ahead, price actual_roll_mean_3.


## 4. Explicarea unei recomandari de optimizare (rezultate reale din Etapa III)

In [3]:
# Re-rezolvam dispatch-ul bateriei pe preturi reale Spania pentru un vector x concret
df = pd.read_parquet(PROJECT_ROOT / "data" / "processed" / "pret_spania_features.parquet")
prices = df["price actual"].iloc[-24:].values.astype(float)  # o zi
cfg = BatteryConfig(capacity=10, p_max=2.5, soc_init=0.5, lambda_deg=0.02, cyclic=True)
out = solve_battery_dispatch(prices, cfg)

rec = ins.summarize_optimization(kind="battery", prices=prices, x=out["x"],
                                 profit=out["profit"]["net_profit"])
print("=== Recomandare baterie (determinist) ===\n")
print(rec["template"])

print("\n=== Recomandare load shifting (determinist) ===\n")
rec2 = ins.summarize_optimization(kind="load_shifting", savings_pct=4.74, peak_reduction_pct=12)
print(rec2["template"])

=== Recomandare baterie (determinist) ===

Recomandare de operare a bateriei: incarca in orele cu pret scazut (02:00, 03:00, 04:00, 05:00, 07:00, 11:00, 13:00, 15:00, 16:00, 22:00, 23:00) si descarca in orele cu pret ridicat (00:00, 01:00, 06:00, 10:00, 12:00, 14:00, 17:00, 18:00, 19:00, 20:00). Aplicand acest plan, profitul estimat pe 24 ore este de aproximativ 167 EUR, respectand limitele fizice ale bateriei.

=== Recomandare load shifting (determinist) ===

Recomandare de gestionare a consumului: muta consumul flexibil din orele de varf (seara) catre orele ieftine (noaptea). Aceasta reorganizare reduce factura cu aproximativ 4.7%, fara a scadea energia totala consumata. Consumul la varf scade cu circa 12%, usurand presiunea pe retea.


## 5. Generarea cu flan-t5 (HuggingFace)

Aici apelam efectiv modelul de limbaj. La **prima rulare in PyCharm**, codul descarca flan-t5 (~1 GB) si genereaza text. Celula e protejata cu try/except: daca transformers/modelul nu sunt disponibile in mediul curent, afiseaza un mesaj clar si foloseste varianta determinista - notebook-ul nu se blocheaza.

Nota onesta: flan-t5-base e un model mic, antrenat preponderent pe engleza. In romana, calitatea poate fi modesta; pentru productie s-ar folosi un model mai mare sau specializat pe romana. Demonstram aici **integrarea reala** a unui LLM open-source in pipeline.

In [4]:
try:
    print("Incarc flan-t5-base (la prima rulare descarca modelul, poate dura)...")
    pipe = ins.get_pipeline("google/flan-t5-base", device="cpu")
    res_llm = ins.explain_prediction(
        dataset="Spanish energy price", target="hourly price (EUR/MWh)",
        metrics=metrics_spania, top_features=["previous-hour price", "day-ahead price", "3h rolling mean"],
        pipe=pipe, use_llm=True, max_new_tokens=120,
    )
    print("\n=== Output flan-t5 ===\n")
    print(res_llm.get("llm", "(fara output LLM)"))
except Exception as e:
    print(f"\n[INFO] flan-t5 nu e disponibil in acest mediu ({type(e).__name__}).")
    print("       Va rula in PyCharm dupa `pip install transformers torch` (prima rulare descarca modelul).")
    print("\nFolosim varianta determinista (fallback), mereu disponibila:\n")
    print(res["template"])

Incarc flan-t5-base (la prima rulare descarca modelul, poate dura)...



[INFO] flan-t5 nu e disponibil in acest mediu (ImportError).
       Va rula in PyCharm dupa `pip install transformers torch` (prima rulare descarca modelul).

Folosim varianta determinista (fallback), mereu disponibila:

Modelul pentru pretul energiei in Spania prezice pretul orar (EUR/MWh) cu o precizie foarte buna, utila in practica. Coeficientul de determinare R2 este 0.970, adica modelul explica 97.0% din variatia tintei. Eroarea medie absoluta este de aproximativ 1.49 EUR/MWh, iar eroarea procentuala medie (MAPE) de 2.5%. Cei mai influenti factori in predictie sunt: price actual_lag_1, price day ahead, price actual_roll_mean_3.


## 6. Concluzii

Componenta LLM inchide lantul decizional al platformei: de la date brute (Etapa I), la predictii (Etapa II), la recomandari optime (Etapa III) si, in final, la **explicatii in limbaj natural** intelese de operatorul uman. Am demonstrat integrarea unui model open-source (flan-t5) prin prompt engineering, sustinuta de un generator determinist care garanteaza un raspuns corect si reproductibil in romana.

Aceasta arhitectura duala reflecta o decizie matura de inginerie: folosim puterea unui LLM acolo unde aduce valoare (formulare naturala, variata), dar ne bazam pe sabloane deterministe pentru a garanta corectitudinea cifrelor si robustetea sistemului. Urmatorul si ultimul pas este Etapa V - aplicatia Streamlit care expune interactiv toate aceste componente intr-o interfata unica.